<a href="https://colab.research.google.com/github/Shashank-Amancha/Reinforcement-Learning/blob/main/Assignment_10(RL).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# States
def env_state():
    return ["S0", "S1"]

states = env_state()
print("States:", states)

States: ['S0', 'S1']


In [ ]:
# Actions
def env_actions():
    return ["up", "down", "left", "Right"]

actions = env_actions()
print("Actions:", actions)

Actions: ['up', 'down', 'left', 'Right']


In [ ]:
# Rewards
def env_reward():
    return {
        "move": -1,
        "goal": 10,
        "obs": -1
    }

rewards = env_reward()
print("Rewards:", rewards)

Rewards: {'move': -1, 'goal': 10, 'obs': -1}


In [ ]:
# Discount Factor
def env_df():
    return 0.9

gamma = env_df()
print("Discount factor:", gamma)

Discount factor: 0.9


In [ ]:
# Transition Mapping
def env_transition():
    return {
        ("S0", "up"): "S0",
        ("S0", "down"): "S1",
        ("S0", "left"): "S0",
        ("S0", "Right"): "S0",
        ("S1", "up"): "S0",
        ("S1", "down"): "S1",
        ("S1", "left"): "S1",
        ("S1", "Right"): "G"
    }

transition = env_transition()
print("Transition mapping:")
for key, value in transition.items():
    print(f"{key} -> {value}")

Transition mapping:
('S0', 'up') -> S0
('S0', 'down') -> S1
('S0', 'left') -> S0
('S0', 'Right') -> S0
('S1', 'up') -> S0
('S1', 'down') -> S1
('S1', 'left') -> S1
('S1', 'Right') -> G


In [ ]:
# Reward for a State-Action Transition
def get_reward(state, action, next_state):
    if next_state == "G":
        return rewards["goal"]
    return rewards["move"]

In [ ]:
# Inspect the MDP
print("States:", states)
print("Actions:", actions)
print("Rewards:", rewards)
print("Discount factor:", gamma)
print("Terminal state: G")
print("V(G) = 0")

States: ['S0', 'S1']
Actions: ['up', 'down', 'left', 'Right']
Rewards: {'move': -1, 'goal': 10, 'obs': -1}
Discount factor: 0.9
Terminal state: G
V(G) = 0


In [ ]:
# Verify all transitions
print("Transition verification:")
for state in states:
    for action in actions:
        next_state = transition[(state, action)]
        reward = get_reward(state, action, next_state)
        print(
            f"Current state: {state}, "
            f"Action: {action}, "
            f"Next state: {next_state}, "
            f"Immediate reward: {reward}"
        )

assert all((state, action) in transition for state in states for action in actions)
print("\nEvery state-action pair has a defined transition.")

Transition verification:
Current state: S0, Action: up, Next state: S0, Immediate reward: -1
Current state: S0, Action: down, Next state: S1, Immediate reward: -1
Current state: S0, Action: left, Next state: S0, Immediate reward: -1
Current state: S0, Action: Right, Next state: S0, Immediate reward: -1
Current state: S1, Action: up, Next state: S0, Immediate reward: -1
Current state: S1, Action: down, Next state: S1, Immediate reward: -1
Current state: S1, Action: left, Next state: S1, Immediate reward: -1
Current state: S1, Action: Right, Next state: G, Immediate reward: 10

Every state-action pair has a defined transition.


In [ ]:
# Initialize State Values
V = {state: 0.0 for state in states}
V["G"] = 0.0

print("Initial state values:")
for state in states:
    print(f"V0({state}) = {V[state]}")
print(f"V(G) = {V['G']}")

Initial state values:
V0(S0) = 0.0
V0(S1) = 0.0
V(G) = 0.0


In [ ]:
# Value Iteration (Synchronous Updates)
theta = 1e-8
max_iterations = 100
iteration_history = [V.copy()]

print("Iteration 0:", V)

for iteration in range(1, max_iterations + 1):
    old_V = V.copy()
    new_V = old_V.copy()

    for state in states:
        q_values = []

        for action in actions:
            next_state = transition[(state, action)]
            reward = get_reward(state, action, next_state)
            q_value = reward + gamma * old_V[next_state]
            q_values.append(q_value)

        new_V[state] = max(q_values)

    delta = max(abs(new_V[state] - old_V[state]) for state in states)
    V = new_V
    iteration_history.append(V.copy())

    print(
        f"Iteration {iteration}: "
        + ", ".join(f"V({state}) = {V[state]:.6f}" for state in states)
        + f" | delta = {delta:.8f}"
    )

    if delta < theta:
        print(f"Converged after {iteration} iterations.")
        break
else:
    print("Maximum iterations reached before convergence.")

Iteration 0: {'S0': 0.0, 'S1': 0.0, 'G': 0.0}
Iteration 1: V(S0) = -1.000000, V(S1) = 10.000000 | delta = 10.00000000
Iteration 2: V(S0) = 8.000000, V(S1) = 10.000000 | delta = 9.00000000
Iteration 3: V(S0) = 8.000000, V(S1) = 10.000000 | delta = 0.00000000
Converged after 3 iterations.


In [ ]:
# Calculate Final Action Values
def calculate_q(state, action, state_values):
    next_state = transition[(state, action)]
    reward = get_reward(state, action, next_state)
    return reward + gamma * state_values[next_state]

Q = {}

for state in states:
    Q[state] = {}
    print(f"\nState {state}:")
    for action in actions:
        Q[state][action] = calculate_q(state, action, V)
        next_state = transition[(state, action)]
        reward = get_reward(state, action, next_state)
        print(
            f"Q*({state}, {action}) = "
            f"{reward} + {gamma} * V({next_state}) = {Q[state][action]:.6f}"
        )


State S0:
Q*(S0, up) = -1 + 0.9 * V(S0) = 6.200000
Q*(S0, down) = -1 + 0.9 * V(S1) = 8.000000
Q*(S0, left) = -1 + 0.9 * V(S0) = 6.200000
Q*(S0, Right) = -1 + 0.9 * V(S0) = 6.200000

State S1:
Q*(S1, up) = -1 + 0.9 * V(S0) = 6.200000
Q*(S1, down) = -1 + 0.9 * V(S1) = 8.000000
Q*(S1, left) = -1 + 0.9 * V(S1) = 8.000000
Q*(S1, Right) = 10 + 0.9 * V(G) = 10.000000


In [ ]:
# Extract the Optimal Policy
optimal_policy = {}

for state in states:
    best_value = max(Q[state].values())
    best_actions = [
        action for action, value in Q[state].items()
        if abs(value - best_value) < 1e-10
    ]
    optimal_policy[state] = best_actions

print("Optimal state values:")
for state in states:
    print(f"V*({state}) = {V[state]:.6f}")

print("\nOptimal policy:")
for state in states:
    print(f"pi*({state}) = {optimal_policy[state]}")

Optimal state values:
V*(S0) = 8.000000
V*(S1) = 10.000000

Optimal policy:
pi*(S0) = ['down']
pi*(S1) = ['Right']


In [ ]:
# Verify Bellman Optimality
print("State | V*(s) | max_a Q*(s,a) | Difference")

for state in states:
    max_q = max(Q[state].values())
    difference = abs(V[state] - max_q)

    print(
        f"{state:5s} | {V[state]:.6f} | "
        f"{max_q:.6f} | {difference:.10f}"
    )

    assert difference < 1e-10

print("\nBellman Optimality verification passed.")

State | V*(s) | max_a Q*(s,a) | Difference
S0    | 8.000000 | 8.000000 | 0.0000000000
S1    | 10.000000 | 10.000000 | 0.0000000000

Bellman Optimality verification passed.
